In [1]:
pip install pdfplumber pandas openpyxl


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# FINAL CODE FOR UNEEK WITH NULL COLUMN
import pandas as pd
import re

# --- Get user inputs ---
po_value = input("Enter the PO value (e.g., PO-106070): ")
item_no = input("Enter the Item No value (e.g., COLVNM04Y): ")
base_serial_start = int(input("Enter base serial start number (e.g., 301133): "))

# --- Read Excel ---
df = pd.read_excel(r'C:\Users\Admin\Downloads\pdf2excel.xlsx', skiprows=2)
selected_columns = ['Serial\nNo', 'Description', 'Stamp', 'Pieces']
df_selected = df[selected_columns].copy()

# --- Clean text ---
df_selected['Description'] = df_selected['Description'].str.replace('\n', ' ', regex=True)
df_selected['Stamp'] = df_selected['Stamp'].str.replace('\n', ' ', regex=True)

# --- Rename columns ---
df_selected.rename(columns={
    'Serial\nNo': 'SerialNo',
    'Description': 'CustomerProductionInstruction',
    'Pieces': 'OrderItemPcs'
}, inplace=True)

# --- Remove header repeats ---
df_selected = df_selected[~df_selected['SerialNo'].isin(['Buyer', 'Serial\nNo'])]

# --- Add base columns ---
df_selected.insert(0, 'SrNo', range(1, len(df_selected) + 1))
SrNo_index = df_selected.columns.get_loc('SrNo')
df_selected.insert(SrNo_index + 1, 'StyleCode', '')
df_selected.insert(SrNo_index + 2, 'ItemSize', '')
df_selected.insert(SrNo_index + 3, 'OrderQty', '10')

# --- Move OrderItemPcs ---
orderqty_index = df_selected.columns.get_loc('OrderQty')
orderitempcs_col = df_selected.pop('OrderItemPcs')
df_selected.insert(orderqty_index + 1, 'OrderItemPcs', orderitempcs_col)

# --- Metal and Tone ---
OrderItemPcs_index = df_selected.columns.get_loc('OrderItemPcs')
df_selected.insert(OrderItemPcs_index + 1, 'Metal', '')

def extract_metal(text):
    if pd.notna(text) and '14KY' in text:
        return 'G14Y'
    return ''

df_selected['Metal'] = df_selected['CustomerProductionInstruction'].apply(extract_metal)
Metal_index = df_selected.columns.get_loc('Metal')
df_selected.insert(Metal_index + 1, 'Tone', '')
df_selected['Tone'] = df_selected['CustomerProductionInstruction'].apply(
    lambda x: 'YG' if pd.notna(x) and '14KY' in x.upper() else ''
)

# --- PO and Ref ---
Tone_index = df_selected.columns.get_loc('Tone')
df_selected.insert(Tone_index + 1, 'ItemPoNo', po_value)
itempono_index = df_selected.columns.get_loc('ItemPoNo')
df_selected.insert(itempono_index + 1, 'ItemRefNo', '')
df_selected.insert(itempono_index + 2, 'StockType', '')
df_selected.insert(itempono_index + 3, 'MakeType', '')

# --- Remarks, StampInstruction ---
CustomerProductionInstruction_index = df_selected.columns.get_loc('CustomerProductionInstruction')
df_selected.insert(CustomerProductionInstruction_index + 1, 'SpecialRemarks', '')
df_selected.insert(CustomerProductionInstruction_index + 2, 'DesignProductionInstruction', '')
df_selected.insert(CustomerProductionInstruction_index + 3, 'StampInstruction', '')

# --- Order Group and SKU ---
Stamp_index = df_selected.columns.get_loc('Stamp')
df_selected.insert(Stamp_index + 1, 'OrderGroup', '')
df_selected.insert(Stamp_index + 2, 'Certificate', '')
df_selected.insert(Stamp_index + 3, 'SKUNo', item_no)

# --- Extra fields ---
sku_index = df_selected.columns.get_loc('SKUNo')
new_columns = [
    'Basestoneminwt', 'Basestonemaxwt', 'Basemetalminwt', 'Basemetalmaxwt',
    'Productiondeliverydate', 'Expecteddeliverydate', 'null', 'SetPrice', 'StoneQuality'
]
for i, col in enumerate(new_columns):
    df_selected.insert(sku_index + 1 + i, col, '')

# --- StyleCode + ItemSize ---
def generate_style_and_size(row):
    """Generate StyleCode and ItemSize separately."""
    if pd.notna(row['CustomerProductionInstruction']) and '18IN' in row['CustomerProductionInstruction']:
        tone = row['Tone'] if pd.notna(row['Tone']) else ''
        sku = row['SKUNo'] if pd.notna(row['SKUNo']) else ''
        suffix = 'CO' if 'CO' in sku else ''
        full_style = f"XK2807G-18IN{tone}{suffix}"

        # Split logic: only keep base 'XK2807G' in StyleCode, move '18IN' to ItemSize
        base_style = 'XK2807G'
        item_size = '18 INCH'
        return pd.Series([base_style, item_size])

    return pd.Series(['', ''])

df_selected[['StyleCode', 'ItemSize']] = df_selected.apply(generate_style_and_size, axis=1)

# --- SpecialRemarks ---
def generate_special_remarks(row):
    remarks = []
    sku = row['SKUNo']
    desc = row['CustomerProductionInstruction']
    if pd.notna(sku): remarks.append(sku)
    if pd.notna(desc) and '14KY' in desc: remarks.append('14K YELLOW GOLD')
    if pd.notna(desc) and '18IN' in desc: remarks.append('SZ 18 INCH')
    remarks.append('DIA QLTY-HI-VS')
    return ','.join(remarks)

df_selected['SpecialRemarks'] = df_selected.apply(generate_special_remarks, axis=1)

# --- StampInstruction (group-wise) ---
def generate_stamp_instructions(df, base_serial_start):
    stamp_instructions = []
    for idx, row in df.iterrows():
        srno = row['SrNo']
        start_serial = base_serial_start + (srno - 1) * 10
        end_serial = start_serial + 9

        has_ufjc = 'UFJC' in str(row['Stamp'])
        has_14ky = '14KY' in str(row['CustomerProductionInstruction'])
        qty_is_10 = str(row['OrderQty']) == '10'
        ctw_match = re.search(r'\d+\.\d+CTW', str(row['CustomerProductionInstruction']))
        ctw_value = ctw_match.group() if ctw_match else ''

        if has_ufjc and has_14ky and qty_is_10 and ctw_value:
            instruction = f"UFJC 14KY {start_serial} to {end_serial} {ctw_value}"
        else:
            instruction = ''
        stamp_instructions.append(instruction)
    return stamp_instructions

df_selected['StampInstruction'] = generate_stamp_instructions(df_selected, base_serial_start)

# --- Drop unnecessary columns ---
df_selected.drop(columns=['SerialNo', 'Stamp'], inplace=True)

# --- Display final DataFrame ---
df_selected

# --- Save to Excel ---
df_selected.to_excel("GATI_FORMAT_UNEEK_EDIT.xlsx", index=False)

Enter the PO value (e.g., PO-106070):  PO-106070
Enter the Item No value (e.g., COLVNM04Y):  COLVNM04Y
Enter base serial start number (e.g., 301133):  301133
